In [1]:
!pip install -q requests beautifulsoup4 lxml pandas openpyxl tqdm

In [8]:
# ============================================================
# CAFELAND.VN CRAWLER
#
# - Chỉ crawl tin bán tại TP. Hồ Chí Minh
# - Tự động qua các trang
# - Lấy dữ liệu từ TRANG CHI TIẾT
# - Không lấy nhầm giá/diện tích từ FILTER
# - Suy diễn loại hình BĐS
# - Multithread
# - Checkpoint CSV + JSON
# - Chống trùng URL
# - Ctrl + C vẫn lưu dữ liệu
# ============================================================

import os
import re
import json
import time
import random
import threading
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIG
# ============================================================

ROW = 20000

START_PAGE = 1

BASE_URL = (
    "https://nhadat.cafeland.vn/"
    "nha-dat-ban-tai-tp-ho-chi-minh/"
)

CHECKPOINT_FILE = "checkpoint_cafeland.csv"
STATE_FILE = "checkpoint_cafeland_state.json"

OUTPUT_CSV = "cafeland.csv"
OUTPUT_XLSX = "cafeland.xlsx"

COLUMNS = [
    "Tieu_de",
    "Gia_ban",
    "Dien_tich",
    "Dia_diem",
    "Loai_hinh_BDS",
    "Mo_ta_Dac_diem",
    "Nguoi_dang_Chu_dau_tu",
    "Ngay_dang",
    "URL"
]

MAX_WORKERS = 12

RETRIES = 3

TIMEOUT = 20

CHECKPOINT_EVERY = 100


# ============================================================
# 2. THREAD LOCAL SESSION
# ============================================================

_thread_local = threading.local()


def get_session():

    if not hasattr(_thread_local, "session"):

        session = requests.Session()

        retry_strategy = Retry(
            total=RETRIES,
            connect=RETRIES,
            read=RETRIES,
            status=RETRIES,
            backoff_factor=0.5,

            status_forcelist=[
                429,
                500,
                502,
                503,
                504
            ],

            allowed_methods=[
                "GET"
            ]
        )

        adapter = HTTPAdapter(
            max_retries=retry_strategy,
            pool_connections=MAX_WORKERS,
            pool_maxsize=MAX_WORKERS
        )

        session.mount(
            "https://",
            adapter
        )

        session.mount(
            "http://",
            adapter
        )

        session.headers.update({

            "User-Agent":
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/151.0.0.0 Safari/537.36",

            "Accept-Language":
                "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",

            "Accept":
                "text/html,application/xhtml+xml,"
                "application/xml;q=0.9,image/avif,image/webp,"
                "*/*;q=0.8"
        })

        _thread_local.session = session

    return _thread_local.session


# ============================================================
# 3. GET HTML
# ============================================================

def get_soup(url):

    try:

        response = get_session().get(
            url,
            timeout=TIMEOUT
        )

        if response.status_code != 200:
            return None

        return BeautifulSoup(
            response.content,
            "lxml"
        )

    except Exception:
        return None


# ============================================================
# 4. CLEAN TEXT
# ============================================================

def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def normalize_url(url):

    if not url:
        return ""

    url = url.strip()

    if url.startswith("//"):
        url = "https:" + url

    elif url.startswith("/"):
        url = "https://nhadat.cafeland.vn" + url

    elif not url.startswith("http"):
        url = "https://nhadat.cafeland.vn/" + url

    return url.split("#")[0]


# ============================================================
# 5. TITLE
# ============================================================

def extract_title(soup):

    h1 = soup.find("h1")

    if h1:

        value = clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )

        if value:
            return value

    title = soup.find("title")

    if title:

        value = clean_text(
            title.get_text(
                " ",
                strip=True
            )
        )

        value = re.sub(
            r"\s*\|\s*CafeLand.*$",
            "",
            value,
            flags=re.I
        )

        return value

    return ""


# ============================================================
# 6. DETAIL CONTAINER
#
# Ưu tiên vùng:
# .container.dangtin
#
# Nếu website thay đổi HTML thì fallback về body.
# ============================================================

def get_detail_region(soup):

    selectors = [

        ".container.dangtin",

        ".dangtin",

        ".container .dangtin",

        ".detail-content",

        ".content-detail"
    ]

    for selector in selectors:

        node = soup.select_one(
            selector
        )

        if node:

            text = clean_text(
                node.get_text(
                    " ",
                    strip=True
                )
            )

            if len(text) > 100:

                return node

    return soup


# ============================================================
# 7. FIND VALUE BY LABEL
# ============================================================

def find_label_value(
    region,
    label,
    stop_labels=None
):

    if stop_labels is None:

        stop_labels = []

    label_lower = label.lower()

    # --------------------------------------------------------
    # Cách 1: tìm text node đúng bằng label
    # --------------------------------------------------------

    for node in region.find_all(
        string=True
    ):

        current = clean_text(
            node
        )

        if current.lower() != label_lower:
            continue

        parent = node.parent

        if not parent:
            continue

        # ----------------------------------------------------
        # Cùng parent
        # ----------------------------------------------------

        parent_text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        if len(parent_text) > len(label) + 3:

            value = re.sub(
                rf"^{re.escape(label)}\s*:?\s*",
                "",
                parent_text,
                flags=re.I
            )

            value = clean_text(
                value
            )

            if value:

                if not any(
                    stop.lower() in value.lower()
                    for stop in stop_labels
                ):

                    return value

        # ----------------------------------------------------
        # Sibling
        # ----------------------------------------------------

        for sibling in parent.find_next_siblings():

            value = clean_text(
                sibling.get_text(
                    " ",
                    strip=True
                )
            )

            if not value:
                continue

            if any(
                stop.lower() == value.lower()
                for stop in stop_labels
            ):
                break

            return value

    return ""


# ============================================================
# 8. GIÁ BÁN
#
# Trang chi tiết CafeLand có:
#
# Giá bán
# 1 tỷ 930 triệu
# 35,9 triệu/m²
#
# Ta chỉ lấy giá bán đầu tiên,
# không lấy giá/m2.
# ============================================================

def extract_price(soup):

    region = get_detail_region(
        soup
    )

    # --------------------------------------------------------
    # Tìm label Giá bán
    # --------------------------------------------------------

    for node in region.find_all(
        string=True
    ):

        text = clean_text(
            node
        )

        if text.lower() != "giá bán":
            continue

        parent = node.parent

        if not parent:
            continue

        # ----------------------------------------------------
        # Tìm trong parent
        # ----------------------------------------------------

        parent_text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        value = re.sub(
            r"^Giá bán\s*:?\s*",
            "",
            parent_text,
            flags=re.I
        )

        value = clean_text(
            value
        )

        if (
            value
            and re.search(r"\d", value)
            and "m²" not in value.lower()
            and "m2" not in value.lower()
        ):

            return value

        # ----------------------------------------------------
        # Tìm sibling
        # ----------------------------------------------------

        for sibling in parent.find_next_siblings():

            value = clean_text(
                sibling.get_text(
                    " ",
                    strip=True
                )
            )

            if not value:
                continue

            if re.search(
                r"\d",
                value
            ):

                if (
                    "triệu/m²" not in value.lower()
                    and "triệu/m2" not in value.lower()
                ):

                    return value

    # --------------------------------------------------------
    # Fallback regex trong detail region
    # --------------------------------------------------------

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )

    match = re.search(
        r"Giá bán\s+"
        r"(.{1,100}?)"
        r"(?=\s+\d[\d.,]*\s*triệu/m[²2]|\s+Diện tích)",
        text,
        flags=re.I
    )

    if match:

        value = clean_text(
            match.group(1)
        )

        if value:

            return value

    return ""


# ============================================================
# 9. DIỆN TÍCH
# ============================================================

def extract_area(soup):

    region = get_detail_region(
        soup
    )

    # --------------------------------------------------------
    # Tìm label Diện tích
    # --------------------------------------------------------

    for node in region.find_all(
        string=True
    ):

        text = clean_text(
            node
        )

        if text.lower() != "diện tích":
            continue

        parent = node.parent

        if not parent:
            continue

        parent_text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        match = re.search(
            r"Diện tích\s*:?\s*"
            r"([\d.,]+\s*m(?:²|2))",
            parent_text,
            flags=re.I
        )

        if match:

            return clean_text(
                match.group(1)
            )

        # ----------------------------------------------------
        # sibling
        # ----------------------------------------------------

        for sibling in parent.find_next_siblings():

            value = clean_text(
                sibling.get_text(
                    " ",
                    strip=True
                )
            )

            match = re.search(
                r"([\d.,]+\s*m(?:²|2))",
                value,
                flags=re.I
            )

            if match:

                return clean_text(
                    match.group(1)
                )

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )

    match = re.search(
        r"Diện tích\s*:?\s*"
        r"([\d.,]+\s*m(?:²|2))",
        text,
        flags=re.I
    )

    if match:

        return clean_text(
            match.group(1)
        )

    return ""


# ============================================================
# 10. ĐỊA ĐIỂM
#
# Mục tiêu:
# chỉ trả về TP. Hồ Chí Minh
# ============================================================

def extract_location(soup):

    region = get_detail_region(
        soup
    )

    # --------------------------------------------------------
    # CafeLand hiện có:
    #
    # Vị trí:
    # Lái Thiêu
    # TP. Hồ Chí Minh
    # --------------------------------------------------------

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )

    # --------------------------------------------------------
    # Vì crawler đang crawl URL TP.HCM,
    # chỉ cần xác nhận địa điểm thuộc HCM.
    # --------------------------------------------------------

    hcm_patterns = [

        "tp. hồ chí minh",
        "tp hồ chí minh",
        "tp.hồ chí minh",
        "thành phố hồ chí minh",
        "hồ chí minh",
        "tphcm",
        "tp hcm",
        "tp.hcm",
        "sài gòn",
        "saigon"
    ]

    text_lower = text.lower()

    for pattern in hcm_patterns:

        if pattern in text_lower:

            return "TP. Hồ Chí Minh"

    return ""


# ============================================================
# 11. LOẠI HÌNH BĐS
#
# Ưu tiên:
# URL → TITLE → "Loại địa ốc" → DESCRIPTION
# ============================================================

def extract_property_type(
    soup,
    title,
    description,
    url
):

    title_lower = clean_text(
        title
    ).lower()

    description_lower = clean_text(
        description
    ).lower()

    url_lower = clean_text(
        url
    ).lower()

    # --------------------------------------------------------
    # URL
    # --------------------------------------------------------

    url_patterns = [

        ("biet-thu", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("can-ho", "Căn hộ"),
        ("chung-cu", "Căn hộ"),

        ("nha-rieng", "Nhà riêng"),
        ("nha-pho", "Nhà phố"),
        ("nha-mat-pho", "Nhà mặt phố"),

        ("dat-nen", "Đất nền"),
        ("dat-tho-cu", "Đất thổ cư"),

        ("shophouse", "Shophouse"),

        ("nha-hang", "Nhà hàng - Khách sạn"),
        ("khach-san", "Nhà hàng - Khách sạn"),

        ("kho-nha-xuong", "Kho - Nhà xưởng"),
        ("nha-xuong", "Kho - Nhà xưởng")
    ]

    for keyword, result in url_patterns:

        if keyword in url_lower:

            return result

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title_patterns = [

        ("biệt thự", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("căn hộ", "Căn hộ"),
        ("chung cư", "Căn hộ"),

        ("nhà mặt phố", "Nhà mặt phố"),
        ("nhà mặt tiền", "Nhà mặt tiền"),

        ("nhà phố", "Nhà phố"),
        ("nhà riêng", "Nhà riêng"),

        ("đất nền", "Đất nền"),
        ("đất thổ cư", "Đất thổ cư"),

        ("nhà xưởng", "Kho - Nhà xưởng"),
        ("kho xưởng", "Kho - Nhà xưởng"),

        ("khách sạn", "Nhà hàng - Khách sạn")
    ]

    for keyword, result in title_patterns:

        if keyword in title_lower:

            return result

    # --------------------------------------------------------
    # LOẠI ĐỊA ỐC
    # --------------------------------------------------------

    region = get_detail_region(
        soup
    )

    for node in region.find_all(
        string=True
    ):

        text = clean_text(
            node
        )

        if text.lower() != "loại địa ốc":
            continue

        parent = node.parent

        if not parent:
            continue

        parent_text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        value = re.sub(
            r"^Loại địa ốc\s*:?\s*",
            "",
            parent_text,
            flags=re.I
        )

        value = clean_text(
            value
        )

        if value:

            return value

        for sibling in parent.find_next_siblings():

            value = clean_text(
                sibling.get_text(
                    " ",
                    strip=True
                )
            )

            if value:

                return value

    # --------------------------------------------------------
    # DESCRIPTION
    # --------------------------------------------------------

    description_patterns = [

        ("biệt thự", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("căn hộ", "Căn hộ"),
        ("chung cư", "Căn hộ"),

        ("nhà mặt phố", "Nhà mặt phố"),
        ("nhà mặt tiền", "Nhà mặt tiền"),

        ("nhà phố", "Nhà phố"),
        ("nhà riêng", "Nhà riêng"),

        ("đất nền", "Đất nền"),
        ("đất thổ cư", "Đất thổ cư"),

        ("nhà xưởng", "Kho - Nhà xưởng"),
        ("kho xưởng", "Kho - Nhà xưởng"),

        ("khách sạn", "Nhà hàng - Khách sạn")
    ]

    for keyword, result in description_patterns:

        if keyword in description_lower:

            return result

    # --------------------------------------------------------
    # Tổng quát
    # --------------------------------------------------------

    if re.search(
        r"\bbán nhà\b",
        title_lower
    ):

        return "Nhà"

    if re.search(
        r"\bbán đất\b",
        title_lower
    ):

        return "Đất"

    return ""


# ============================================================
# 12. MÔ TẢ
# ============================================================

def extract_description(soup):

    region = get_detail_region(
        soup
    )

    # --------------------------------------------------------
    # Tìm heading "Thông tin mô tả"
    # --------------------------------------------------------

    for node in region.find_all(
        string=True
    ):

        text = clean_text(
            node
        )

        if text.lower() != "thông tin mô tả":
            continue

        parent = node.parent

        if not parent:
            continue

        # ----------------------------------------------------
        # Thử lấy container cha
        # ----------------------------------------------------

        container = parent.parent

        if container:

            value = clean_text(
                container.get_text(
                    " ",
                    strip=True
                )
            )

            value = re.sub(
                r"^Thông tin mô tả\s*",
                "",
                value,
                flags=re.I
            )

            # Cắt trước các phần phía sau
            value = re.split(
                r"\s+Kiến trúc\s*-\s*Tiện ích\s+",
                value,
                flags=re.I
            )[0]

            value = re.split(
                r"\s+Loại địa ốc\s+",
                value,
                flags=re.I
            )[0]

            if len(value) >= 20:

                return value[:12000]

    # --------------------------------------------------------
    # fallback selector
    # --------------------------------------------------------

    selectors = [

        ".description",
        ".detail-content",
        ".content-detail",
        ".post-content",
        ".slide-description"
    ]

    for selector in selectors:

        node = region.select_one(
            selector
        )

        if node:

            value = clean_text(
                node.get_text(
                    " ",
                    strip=True
                )
            )

            if len(value) >= 20:

                return value[:12000]

    return ""


# ============================================================
# 13. NGƯỜI ĐĂNG
# ============================================================

def extract_owner(soup):

    region = get_detail_region(soup)

    # ========================================================
    # 1. LẤY TOÀN BỘ TEXT TRONG DETAIL
    # ========================================================

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )

    # ========================================================
    # 2. ƯU TIÊN CÁC MẪU CÓ LABEL NGƯỜI ĐĂNG
    # ========================================================

    patterns = [

        r"Được đăng bởi\s*:?\s*"
        r"(.{2,100}?)"
        r"(?=\s+\d{7,12}|\s+Hiện số|\s+Gửi email|\s+Liên hệ)",

        r"Đăng bởi\s*:?\s*"
        r"(.{2,100}?)"
        r"(?=\s+\d{7,12}|\s+Hiện số|\s+Gửi email|\s+Liên hệ)",

        r"Người đăng\s*:?\s*"
        r"(.{2,100}?)"
        r"(?=\s+\d{7,12}|\s+Hiện số|\s+Gửi email|\s+Liên hệ)",

        r"Liên hệ\s*:?\s*"
        r"(.{2,100}?)"
        r"(?=\s+\d{7,12}|\s+Hiện số|\s+Gửi email)"
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.I
        )

        if not match:
            continue

        value = clean_text(
            match.group(1)
        )

        # ====================================================
        # LOẠI CÁC TEXT RÁC
        # ====================================================

        if not value:
            continue

        bad_keywords = [

            "cần mua bất động sản",
            "đăng nhu cầu tìm mua",
            "đăng nhu cầu để người bán liên hệ",
            "đăng tin",
            "đăng ký",
            "đăng nhập",
            "xem thêm",
            "lưu tin",
            "chia sẻ",
            "bản đồ",
            "gửi email",
            "hiện số"
        ]

        if any(
            keyword in value.lower()
            for keyword in bad_keywords
        ):
            continue

        # Không nhận chuỗi quá dài
        if len(value) > 100:
            continue

        # Không nhận chuỗi chứa quá nhiều câu
        if value.count(".") > 2:
            continue

        return value

    # ========================================================
    # 3. KHÔNG CÓ LABEL → KHÔNG ĐOÁN
    # ========================================================

    return ""

# ============================================================
# 14. NGÀY ĐĂNG
# ============================================================

def extract_post_date(soup):

    region = get_detail_region(
        soup
    )

    text = clean_text(
        region.get_text(
            " ",
            strip=True
        )
    )

    # Ngày đăng: 21-04-2026
    match = re.search(
        r"Ngày đăng\s*:?\s*"
        r"(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
        text,
        flags=re.I
    )

    if match:

        return match.group(1)

    # Cập nhật: ...
    match = re.search(
        r"Cập nhật\s*:?\s*"
        r"(\d{1,2}[-/]\d{1,2}[-/]\d{4})",
        text,
        flags=re.I
    )

    if match:

        return match.group(1)

    return ""


# ============================================================
# 15. SCRAPE DETAIL
# ============================================================

def scrape_detail(url):

    soup = get_soup(
        url
    )

    if soup is None:
        return None

    try:

        title = extract_title(
            soup
        )

        if not title:
            return None

        price = extract_price(
            soup
        )

        area = extract_area(
            soup
        )

        location = extract_location(
            soup
        )

        description = extract_description(
            soup
        )

        property_type = extract_property_type(
            soup,
            title,
            description,
            url
        )

        owner = extract_owner(
            soup
        )

        post_date = extract_post_date(
            soup
        )

        return {

            "Tieu_de":
                title,

            "Gia_ban":
                price,

            "Dien_tich":
                area,

            "Dia_diem":
                location,

            "Loai_hinh_BDS":
                property_type,

            "Mo_ta_Dac_diem":
                description,

            "Nguoi_dang_Chu_dau_tu":
                owner,

            "Ngay_dang":
                post_date,

            "URL":
                url
        }

    except Exception:

        return None


# ============================================================
# 16. GET DETAIL URLS
# ============================================================

def get_detail_urls(page):

    if page == 1:

        url = BASE_URL

    else:

        url = (
            BASE_URL.rstrip("/")
            + f"/page-{page}/"
        )

    soup = get_soup(
        url
    )

    if soup is None:

        return []

    urls = []

    seen = set()

    # --------------------------------------------------------
    # Tìm tất cả link
    # --------------------------------------------------------

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = normalize_url(
            a.get("href")
        )

        if not href:
            continue

        # ----------------------------------------------------
        # Chỉ lấy link thuộc nhadat.cafeland.vn
        # ----------------------------------------------------

        if "nhadat.cafeland.vn" not in href:
            continue

        # ----------------------------------------------------
        # Loại bỏ link category/filter
        # ----------------------------------------------------

        excluded = [

            "/nha-dat-ban/",
            "/nha-dat-ban-tai-",
            "/cho-thue",
            "/du-an",
            "/blog",
            "/tim-kiem",
            "/dang-tin",
            "/dang-ky",
            "/dang-nhap"
        ]

        if any(
            x in href.lower()
            for x in excluded
        ):

            continue

        # ----------------------------------------------------
        # Tin chi tiết CafeLand thường kết thúc:
        # - số ID .html
        # ----------------------------------------------------

        if not re.search(
            r"-\d{6,}\.html$",
            href,
            flags=re.I
        ):

            continue

        if href in seen:
            continue

        seen.add(
            href
        )

        urls.append(
            href
        )

    return urls


# ============================================================
# 17. VALIDATE
# ============================================================

def validate_row(row):

    if not row:
        return False

    if not row.get(
        "URL",
        ""
    ):
        return False

    if not row.get(
        "Tieu_de",
        ""
    ):
        return False

    return True


# ============================================================
# 18. SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    rows,
    current_page
):

    if not rows:
        return

    df = pd.DataFrame(
        rows,
        columns=COLUMNS
    )

    df = df.fillna("")

    df = df.drop_duplicates(
        subset=["URL"],
        keep="first"
    )

    df.to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    state = {

        "current_page":
            current_page,

        "rows":
            len(df),

        "last_saved":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }

    with open(
        STATE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# 19. LOAD CHECKPOINT
# ============================================================

def load_checkpoint():

    if not os.path.exists(
        CHECKPOINT_FILE
    ):

        return [], set(), START_PAGE

    try:

        df = pd.read_csv(
            CHECKPOINT_FILE,
            encoding="utf-8-sig",
            dtype=str
        )

        df = df.fillna("")

        for col in COLUMNS:

            if col not in df.columns:

                df[col] = ""

        df = df[
            COLUMNS
        ]

        df = df.drop_duplicates(
            subset=["URL"],
            keep="first"
        )

        rows = df.to_dict(
            orient="records"
        )

        urls = set(
            df["URL"].tolist()
        )

        current_page = START_PAGE

        if os.path.exists(
            STATE_FILE
        ):

            try:

                with open(
                    STATE_FILE,
                    "r",
                    encoding="utf-8"
                ) as f:

                    state = json.load(
                        f
                    )

                current_page = int(
                    state.get(
                        "current_page",
                        START_PAGE
                    )
                )

            except Exception:

                pass

        return (
            rows,
            urls,
            current_page
        )

    except Exception:

        return [], set(), START_PAGE


# ============================================================
# 20. MAIN CRAWLER
# ============================================================

def crawl_cafeland():

    rows, scraped_urls, current_page = (
        load_checkpoint()
    )

    print(
        "=" * 70
    )

    print(
        "CAFELAND.VN CRAWLER"
    )

    print(
        "=" * 70
    )

    print(
        f"Mục tiêu        : {ROW:,}"
    )

    print(
        f"Checkpoint      : {len(rows):,}"
    )

    print(
        f"Bắt đầu page    : {current_page}"
    )

    print(
        f"Workers         : {MAX_WORKERS}"
    )

    print(
        "Khu vực         : TP. Hồ Chí Minh"
    )

    print(
        "=" * 70
    )

    last_saved = len(
        rows
    )

    empty_pages = 0

    try:

        with ThreadPoolExecutor(
            max_workers=MAX_WORKERS
        ) as executor:

            while len(rows) < ROW:

                print(
                    f"\nPAGE {current_page}"
                )

                page_urls = get_detail_urls(
                    current_page
                )

                print(
                    f"URL tìm thấy: {len(page_urls)}"
                )

                if not page_urls:

                    empty_pages += 1

                    print(
                        f"Trang rỗng: {empty_pages}"
                    )

                    # tránh chạy vô hạn nếu URL phân trang
                    # không còn tồn tại

                    if empty_pages >= 3:

                        print(
                            "Không còn page có dữ liệu."
                        )

                        break

                    current_page += 1

                    continue

                empty_pages = 0

                new_urls = [

                    url

                    for url in page_urls

                    if url not in scraped_urls
                ]

                print(
                    f"URL mới: {len(new_urls)}"
                )

                if not new_urls:

                    current_page += 1

                    continue

                remaining = (
                    ROW
                    -
                    len(rows)
                )

                new_urls = new_urls[
                    :remaining
                ]

                futures = {

                    executor.submit(
                        scrape_detail,
                        url
                    ):
                    url

                    for url in new_urls
                }

                page_rows = []

                with tqdm(
                    total=len(futures),
                    desc=f"Page {current_page}",
                    unit="tin"
                ) as pbar:

                    for future in as_completed(
                        futures
                    ):

                        url = futures[
                            future
                        ]

                        try:

                            result = future.result()

                            if (
                                result
                                and
                                validate_row(
                                    result
                                )
                            ):

                                page_rows.append(
                                    result
                                )

                                scraped_urls.add(
                                    url
                                )

                        except Exception:

                            pass

                        pbar.update(
                            1
                        )

                rows.extend(
                    page_rows
                )

                # ------------------------------------------------
                # Chống trùng
                # ------------------------------------------------

                unique = {}

                for row in rows:

                    url = row.get(
                        "URL",
                        ""
                    )

                    if url:

                        unique[url] = row

                rows = list(
                    unique.values()
                )

                rows = rows[
                    :ROW
                ]

                print(
                    f"Đã lấy: {len(rows):,}/{ROW:,}"
                )

                # ------------------------------------------------
                # SAVE CHECKPOINT
                # ------------------------------------------------

                if (
                    len(rows) - last_saved
                    >= CHECKPOINT_EVERY
                    or page_rows
                ):

                    save_checkpoint(
                        rows,
                        current_page + 1
                    )

                    last_saved = len(
                        rows
                    )

                    print(
                        "✓ Đã lưu checkpoint"
                    )

                current_page += 1

                time.sleep(
                    random.uniform(
                        0.2,
                        0.5
                    )
                )

    except KeyboardInterrupt:

        print(
            "\n\nĐÃ DỪNG CRAWLER BẰNG CTRL + C"
        )

        save_checkpoint(
            rows,
            current_page
        )

        print(
            f"✓ Đã lưu {len(rows):,} dòng"
        )

        return pd.DataFrame(
            rows,
            columns=COLUMNS
        )

    except Exception as e:

        print(
            "\nCrawler lỗi:",
            repr(e)
        )

        save_checkpoint(
            rows,
            current_page
        )

        return pd.DataFrame(
            rows,
            columns=COLUMNS
        )

    # ========================================================
    # FINAL DATA
    # ========================================================

    df = pd.DataFrame(
        rows,
        columns=COLUMNS
    )

    df = df.fillna("")

    df = df.drop_duplicates(
        subset=["URL"],
        keep="first"
    )

    df = df.head(
        ROW
    )

    df = df.reset_index(
        drop=True
    )

    # ========================================================
    # SAVE CSV
    # ========================================================

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # SAVE EXCEL
    # ========================================================

    df.to_excel(
        OUTPUT_XLSX,
        index=False
    )

    # ========================================================
    # FINAL CHECKPOINT
    # ========================================================

    save_checkpoint(
        df.to_dict(
            orient="records"
        ),
        current_page
    )

    # ========================================================
    # REPORT
    # ========================================================

    print(
        "\n"
        + "=" * 70
    )

    print(
        "CRAWL HOÀN TẤT"
    )

    print(
        "=" * 70
    )

    print(
        f"Số dòng: {len(df):,}"
    )

    print(
        f"Số cột: {len(df.columns)}"
    )

    print(
        f"CSV: {OUTPUT_CSV}"
    )

    print(
        f"Excel: {OUTPUT_XLSX}"
    )

    print(
        f"Checkpoint: {CHECKPOINT_FILE}"
    )

    print(
        "\nSố lượng thiếu:"
    )

    print(
        df.isna().sum()
    )

    print(
        "\n10 dòng đầu:"
    )

    display(
        df.head(10)
    )

    return df


# ============================================================
# 21. START
# ============================================================

df = crawl_cafeland()

CAFELAND.VN CRAWLER
Mục tiêu        : 20,000
Checkpoint      : 0
Bắt đầu page    : 1
Workers         : 12
Khu vực         : TP. Hồ Chí Minh

PAGE 1
URL tìm thấy: 41
URL mới: 41


Page 1:   0%|          | 0/41 [00:00<?, ?tin/s]

Đã lấy: 41/20,000
✓ Đã lưu checkpoint

PAGE 2
URL tìm thấy: 31
URL mới: 27


Page 2:   0%|          | 0/27 [00:00<?, ?tin/s]

Đã lấy: 68/20,000
✓ Đã lưu checkpoint

PAGE 3
URL tìm thấy: 33
URL mới: 28


Page 3:   0%|          | 0/28 [00:00<?, ?tin/s]

... (759 outputs left out)

Page 383:   0%|          | 0/26 [00:00<?, ?tin/s]

Đã lấy: 9,858/20,000
✓ Đã lưu checkpoint

PAGE 384
URL tìm thấy: 43
URL mới: 26


Page 384:   0%|          | 0/26 [00:00<?, ?tin/s]

Đã lấy: 9,883/20,000
✓ Đã lưu checkpoint

PAGE 385
URL tìm thấy: 41
URL mới: 0

PAGE 386
URL tìm thấy: 41
URL mới: 0

PAGE 387
URL tìm thấy: 41
URL mới: 0

PAGE 388
URL tìm thấy: 41
URL mới: 0

PAGE 389
URL tìm thấy: 41
URL mới: 0

PAGE 390
URL tìm thấy: 41
URL mới: 0

PAGE 391
URL tìm thấy: 41
URL mới: 0

PAGE 392
URL tìm thấy: 41
URL mới: 0

PAGE 393
URL tìm thấy: 41
URL mới: 0

PAGE 394
URL tìm thấy: 41
URL mới: 0

PAGE 395
URL tìm thấy: 41
URL mới: 0

PAGE 396
URL tìm thấy: 41
URL mới: 0

PAGE 397
URL tìm thấy: 41
URL mới: 0

PAGE 398
URL tìm thấy: 41
URL mới: 0

PAGE 399
URL tìm thấy: 41
URL mới: 0

PAGE 400
URL tìm thấy: 41
URL mới: 0

PAGE 401
URL tìm thấy: 41
URL mới: 0

PAGE 402
URL tìm thấy: 41
URL mới: 0

PAGE 403
URL tìm thấy: 41
URL mới: 0

PAGE 404
URL tìm thấy: 41
URL mới: 0

PAGE 405
URL tìm thấy: 41
URL mới: 0

PAGE 406
URL tìm thấy: 41
URL mới: 0

PAGE 407
URL tìm thấy: 41
URL mới: 0

PAGE 408
URL tìm thấy: 41
URL mới: 0

PAGE 409
URL tìm thấy: 41
URL mới: 0

PAGE 410